In [2]:
import pandas as pd
import json
import urllib.parse
import numpy as np
import requests
import time

# Nutrition Facts

## Просмотр датафрейма

In [4]:
df = pd.read_csv("data/result.csv")

In [9]:
df.head()

,Unnamed: 0,rating,almond,amaretto,anchovy,anise,apple,apple juice,apricot,artichoke,...,whiskey,white wine,whole wheat,wild rice,wine,yellow squash,yogurt,yuca,zucchini,turkey
0,0,2.500,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,1,4.375,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2,3.750,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,3,5.000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,4,3.125,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [12]:
df.shape

(20052, 364)

## Оставляем только ингридиенты

In [22]:
cols = df.columns.tolist()
clean_ingredients = cols[2:]
print(len(clean_ingredients))
clean_ingredients

362


['almond',
 'amaretto',
 'anchovy',
 'anise',
 'apple',
 'apple juice',
 'apricot',
 'artichoke',
 'arugula',
 'asian pear',
 'asparagus',
 'avocado',
 'bacon',
 'banana',
 'barley',
 'basil',
 'bass',
 'bean',
 'beef',
 'beef rib',
 'beef shank',
 'beef tenderloin',
 'beer',
 'beet',
 'bell pepper',
 'berry',
 'biscuit',
 'bitters',
 'blackberry',
 'blue cheese',
 'blueberry',
 'bok choy',
 'bourbon',
 'bran',
 'brandy',
 'bread',
 'breadcrumbs',
 'brie',
 'brine',
 'brisket',
 'broccoli',
 'broccoli rabe',
 'brown rice',
 'brownie',
 'brussel sprout',
 'buffalo',
 'bulgur',
 'burrito',
 'butter',
 'buttermilk',
 'butternut squash',
 'butterscotch/caramel',
 'cabbage',
 'calvados',
 'campari',
 'candy',
 'cantaloupe',
 'capers',
 'caraway',
 'cardamom',
 'carrot',
 'cashew',
 'cauliflower',
 'caviar',
 'celery',
 'chambord',
 'champagne',
 'chard',
 'chartreuse',
 'cheddar',
 'cheese',
 'cherry',
 'chestnut',
 'chicken',
 'chickpea',
 'chile',
 'chile pepper',
 'chili',
 'chive',
 'ch

## Создаем пустой справочник продуктов, где
## строки - названия, колонки - нутриенты из PDF-таблиц

In [18]:
nutrient_columns = [
    'fat_dv', 'saturated_fat_dv', 'cholesterol_dv', 'carbohydrate_dv',
    'sodium_dv', 'fiber_dv', 'protein_dv', 'added_sugars_dv',
    'vitamin_a_dv', 'vitamin_c_dv', 'vitamin_d_dv', 'vitamin_e_dv', 
    'calcium_dv', 'iron_dv', 'magnesium_dv', 'zinc_dv', 'potassium_dv'
]
## dv - Daily Value «суточная норма»

ingredients_nutrition_df = pd.DataFrame(
    0.0, 
    index=clean_ingredients, 
    columns=nutrient_columns
)

print(ingredients_nutrition_df.shape)

(364, 17)


In [19]:
ingredients_nutrition_df

,fat_dv,saturated_fat_dv,cholesterol_dv,carbohydrate_dv,sodium_dv,fiber_dv,protein_dv,added_sugars_dv,vitamin_a_dv,vitamin_c_dv,vitamin_d_dv,vitamin_e_dv,calcium_dv,iron_dv,magnesium_dv,zinc_dv,potassium_dv
Unnamed: 0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
rating,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
almond,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
amaretto,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
anchovy,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
yellow squash,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
yogurt,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
yuca,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
zucchini,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## Заполнение ingredients_nutrition_df данными

## Собираем данные о пищевой ценности ингредиентов с помощью USDA API

In [ ]:
API_KEY = "iHKfiOpDRY9507PiPft3dyWV5d98qSIcU9ChvO12"
BASE_URL = 'https://api.nal.usda.gov/fdc/v1'

def get_food_nutrients(ingredient_name):
    search_url = f"{BASE_URL}/foods/search"
    params = {
        'api_key': API_KEY,
        'query': ingredient_name,
        'pageSize': 1,
        'dataType': 'Foundation,SR Legacy'
    }
    response = requests.get(search_url, params=params)

    if response.status_code != 200: # 200 - это успех
        print(f"Ошибка API. Статус: {response.status_code}")
        print(f"Текст ошибки: {response.text}")
        return None

    try:
        data = response.json()
    except Exception as e:
        print(f"Не удалось прочитать JSON: {e}")
        print(f"Содержимое ответа: {response.text}")
        return None
    if not data.get('foods'):
        print(f"Продукт '{ingredient_name}' не найден в базе")
        return None

    food = data['foods'][0]
    if 'foodNutrients' in food:
      nutrients = food['foodNutrients']
    else:
      nutrients = []
    result = {}
    for n in nutrients:
        name = n['nutrientName']
        value = n.get('value', 0.0)
        unit = n.get('unitName', 'N/A')
        result[name] = {'value': value, 'unit': unit}

    return result

# тест
test_nutrients = get_food_nutrients('vodka')
print(test_nutrients)

In [ ]:
NUTRIENT_MAP = {
    "Protein": ("protein_dv", 50),
    "Total lipid (fat)": ("fat_dv", 78),
    "Fatty acids, total saturated": ("saturated_fat_dv", 20),
    "Carbohydrate, by difference": ("carbohydrate_dv", 275),
    "Sodium, Na": ("sodium_dv", 2300),
    "Fiber, total dietary": ("fiber_dv", 28),
    "Cholesterol": ("cholesterol_dv", 300),
    "Vitamin A, RAE": ("vitamin_a_dv", 900),
    "Vitamin C, total ascorbic acid": ("vitamin_c_dv", 90),
    "Calcium, Ca": ("calcium_dv", 1300),
    "Iron, Fe": ("iron_dv", 18),
    "Potassium, K": ("potassium_dv", 4700)
}

In [ ]:
for ingredient in clean_ingredients:
    print(f"Обработка: {ingredient}...", end=" ")
    all_nutrients = get_food_nutrients(ingredient)

    if all_nutrients is not None:
      for api_nutrient_name in NUTRIENT_MAP:
        if api_nutrient_name in all_nutrients:
          col_name, daily_norm = NUTRIENT_MAP[api_nutrient_name]
          nutrient_val = all_nutrients[api_nutrient_name]['value']
          percent = nutrient_val / daily_norm * 100
          ingredients_nutrition_df.at[ingredient, col_name] = round(percent, 1)
      print('Готово!')
    else:
      print('Продукт не найден!')
    time.sleep(0.5)

print("\nВсе готово!")
print(ingredients_nutrition_df.head())
ingredients_nutrition_df.to_csv('data/ingredients_with_nutrition.csv')

## Данные сохранены в 'data/ingredients_with_nutrition.csv'

In [27]:
ingredients_with_nutrition_df = pd.read_csv('data/ingredients_with_nutrition.csv')
ingredients_with_nutrition_df.head(10)

,Unnamed: 0,fat_dv,saturated_fat_dv,cholesterol_dv,carbohydrate_dv,sodium_dv,fiber_dv,protein_dv,added_sugars_dv,vitamin_a_dv,vitamin_c_dv,vitamin_d_dv,vitamin_e_dv,calcium_dv,iron_dv,magnesium_dv,zinc_dv,potassium_dv
0,almond,64.4,0.0,0.0,5.9,0.0,33.1,52.4,0.0,0.0,0.0,0.0,0.0,17.8,17.9,0.0,0.0,14.2
1,amaretto,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,anchovy,6.2,6.4,20.0,0.0,4.5,0.0,40.8,0.0,1.7,0.0,0.0,0.0,11.3,18.1,0.0,0.0,8.1
3,anise,20.4,2.9,0.0,18.2,0.7,52.1,35.2,0.0,1.8,23.3,0.0,0.0,49.7,205.6,0.0,0.0,30.6
4,apple,11.2,24.9,10.3,13.5,11.9,8.9,14.8,0.0,10.4,0.6,0.0,0.0,2.3,6.1,0.0,0.0,1.9
5,apple juice,0.1,0.1,0.0,4.3,0.3,0.4,0.0,0.0,0.1,64.3,0.0,0.0,0.3,3.2,0.0,0.0,1.9
6,apricot,0.5,0.1,0.0,4.0,0.0,7.1,2.8,0.0,10.7,11.1,0.0,0.0,1.0,2.2,0.0,0.0,5.5
7,artichoke,0.0,0.0,0.0,6.3,0.2,5.7,4.0,0.0,0.1,4.4,0.0,0.0,1.1,18.9,0.0,0.0,9.1
8,arugula,0.8,0.4,0.0,1.3,1.2,5.7,5.2,0.0,13.2,16.7,0.0,0.0,12.3,8.1,0.0,0.0,7.9
9,asian pear,0.3,0.1,0.0,3.9,0.0,12.9,1.0,0.0,0.0,4.2,0.0,0.0,0.3,0.0,0.0,0.0,2.6


# Similar Recipes

In [25]:
with open('data/full_format_recipes.json', 'r') as f:
    full_recipes = json.load(f)

df_json = pd.DataFrame(full_recipes)
df_json.head()

,directions,fat,date,categories,calories,desc,protein,rating,title,ingredients,sodium
0,"[1. Place the stock, lentils, celery, carrot, ...",7.0,2006-09-01T04:00:00.000Z,"[Sandwich, Bean, Fruit, Tomato, turkey, Vegeta...",426.0,None,30.0,2.500,"Lentil, Apple, and Turkey Wrap","[4 cups low-sodium vegetable or chicken stock,...",559.0
1,[Combine first 9 ingredients in heavy medium s...,23.0,2004-08-20T04:00:00.000Z,"[Food Processor, Onion, Pork, Bake, Bastille D...",403.0,This uses the same ingredients found in boudin...,18.0,4.375,Boudin Blanc Terrine with Red Onion Confit,"[1 1/2 cups whipping cream, 2 medium onions, c...",1439.0
2,[In a large heavy saucepan cook diced fennel a...,7.0,2004-08-20T04:00:00.000Z,"[Soup/Stew, Dairy, Potato, Vegetable, Fennel, ...",165.0,None,6.0,3.750,Potato and Fennel Soup Hodge,"[1 fennel bulb (sometimes called anise), stalk...",165.0
3,[Heat oil in heavy large skillet over medium-h...,NaN,2009-03-27T04:00:00.000Z,"[Fish, Olive, Tomato, Sauté, Low Fat, Low Cal,...",NaN,The Sicilian-style tomato sauce has tons of Me...,NaN,5.000,Mahi-Mahi in Tomato Olive Sauce,"[2 tablespoons extra-virgin olive oil, 1 cup c...",NaN
4,[Preheat oven to 350°F. Lightly grease 8x8x2-i...,32.0,2004-08-20T04:00:00.000Z,"[Cheese, Dairy, Pasta, Vegetable, Side, Bake, ...",547.0,None,20.0,3.125,Spinach Noodle Casserole,"[1 12-ounce package frozen spinach soufflé, th...",452.0


In [26]:
def create_search_url(title):
    if not title or pd.isna(title):
        return None
    clean_title = title.strip()
    query = urllib.parse.quote_plus(clean_title)
    return f"https://www.epicurious.com/search?q={query}"

df_json['url'] = df_json['title'].apply(create_search_url)
df_urls = df_json[['title', 'rating', 'url']].dropna(subset=['title', 'url'])

df_urls = df_urls.drop_duplicates(subset=['title'])
df_urls.to_csv('data/recipes_with_urls.csv', index=False)

print(f"Всего рецептов: {len(df_urls)}")
print(df_urls.head())

Всего рецептов: 17775
                                         title  rating  \
0              Lentil, Apple, and Turkey Wrap    2.500   
1  Boudin Blanc Terrine with Red Onion Confit    4.375   
2                Potato and Fennel Soup Hodge    3.750   
3             Mahi-Mahi in Tomato Olive Sauce    5.000   
4                    Spinach Noodle Casserole    3.125   

                                                 url  
0  https://www.epicurious.com/search?q=Lentil%2C+...  
1  https://www.epicurious.com/search?q=Boudin+Bla...  
2  https://www.epicurious.com/search?q=Potato+and...  
3  https://www.epicurious.com/search?q=Mahi-Mahi+...  
4  https://www.epicurious.com/search?q=Spinach+No...  


## Создаем файл и с ссылками, и с рецептами

In [11]:
urls_df = pd.read_csv('data/recipes_with_urls.csv')
urls_df['title_clean'] = urls_df['title'].str.strip().str.replace('"', '')

with open('data/full_format_recipes.json', 'r') as f:
    json_data = json.load(f)
    
json_df = pd.DataFrame(json_data)
json_df['title_clean'] = json_df['title'].str.strip().str.replace('"', '')
merged = pd.merge(urls_df, json_df[['title_clean', 'ingredients']], on='title_clean', how='left')

def flatten_ingredients(ing_list):
    if isinstance(ing_list, list):
        return " ".join(ing_list).lower()
    return ""

merged['ingredients_text'] = merged['ingredients'].apply(flatten_ingredients)
final_df = merged[['title', 'rating', 'url', 'ingredients_text']]
final_df.to_csv('searchable_recipes.csv', index=False)

In [16]:
df = pd.read_csv('data/searchable_recipes.csv')

df_cleaned = df.drop_duplicates(subset=['title'], keep='first')
df_cleaned['title'] = df_cleaned.title.str.strip()
df_cleaned = df_cleaned.drop_duplicates(subset=['title', 'ingredients_text'], keep='first')
    
df_cleaned.to_csv('data/searchable_recipes.csv', index=False)

/var/folders/4d/x8h05l5x51x8mtcwts51gtm40000gn/T/ipykernel_79588/3644486018.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_cleaned['title'] = df_cleaned.title.str.strip()
